# Binder Design with Cicada

This notebook walks through a standard miniprotein binder design run using Cicada (Psi-1.5 Cicada). Please see the main README for a description of each step and example/README for instructions on how to customize the pipeline steps.

The pipeline consists of four steps:

1. **Featurization** — specify a target structure, cropping, hotspots, and new chain parameters
2. **Simulation** — configure the ODE/SDE sampler and denoising trajectory
3. **Output Configuration** — set output format, directory, and trajectory saving
4. **Run** — verify the configuration and generate binder candidates

## Featurization

Main Arguments
- `name` : A user-readable name that defines this featurization setup.
- Define your target structure with exactly one of the following:
    - `rcsb_record_id` : The RCSB ID of your target; it will be downloaded from the RCSB PDB. This requires an internet connection.
    - `afdb_uniprot_id` : The Uniprot ID of the target; it will be downloaded from AFDB (v6). All isoforms are allowed (e.g. P16871, P16871-2). This requires an internet connection.
    - `structure_path` : Local path to the target structure, in .pdb or .mmcif format.
- Define the chain ID / indexing strategy used for the featurization specification 
    - `index_type` : Which index strategy is used. Options are `original_residue_index`, `auth_residue_index`, described below:
        - `auth_residue_index`: Use the Author-defined indexing: residue indices and chain IDs are defined by the author of the deposited structure file.
        - `original_residue_index`: chains are labeled A, B, C, … and each chain's residues start at 1. 
        
    We recommend loading your structure in [Mol*Viewer](https://molstar.org/viewer/) to select chains and indices. Mol*Viewer explicitly annotates with "AUTH" in scenarios where the author-defined chain ID and residue indexing diverges from standard PDB indexing.

    It is strongly recommended that you check the output featurization before generating designs to make sure that you have defined the correct chain/index parameters, especially if you use `auth_residue_index`. See **'Check Design Pipeline'** below.


Omit `crops` to use the whole structure. `hotspots`, `new_chains_lengths`, and `new_chains_centers_of_mass` must be non-empty. Each accepts multiple configurations. Their Cartesian product yields one featurization per combination.
    
- `crops` : A dictionary of {crop_name : string_crop_range}. Crop ranges are inclusive, e.g. A1-3 corresponds to A1,A2,A3.
- `hotspots` : A dictionary of {hotspot_name : hotspot_positions}.
- `new_chains_lengths` : A list of different chain lengths to generate.
- `new_chains_centers_of_mass` : A dictionary of {center_name : center_of_mass_function}. We recommend you use `PredefinedCenterOfMassEstimator` from `dynopsi.data.featurization.primitives`. See `example/README` for how to pick the center of mass.


In [ ]:
from dynopsi.data.constants import AtomFilter, DSSPLoopFeature, HomooligomerPartnerFeature
from dynopsi.data.featurization import DynoPsiBinderDesignFeaturizationPipeline
from dynopsi.data.featurization.primitives import PredefinedCenterOfMassEstimator

featurization_pipeline = DynoPsiBinderDesignFeaturizationPipeline(
    name="il7ra_3di3_binder_design",
    index_type="auth_residue_index",
    rcsb_record_id="3di3",
    crops={"crop_17_209": "B17-209"},
    hotspots={"hotspot_58V_80L_139Y": "B58,B80,B139"},
    design_conditioning={
        "chain_is_loopy": {"non_loopy": DSSPLoopFeature.NON_LOOP},
        "has_homooligomer_partner": {"no_partner": HomooligomerPartnerFeature.NO_PARTNER},
    },
    new_chains_lengths=[60, 80, 100, 120],
    new_chains_centers_of_mass={"il7center": PredefinedCenterOfMassEstimator(centers_of_mass=[[28.1, 41.4, 47.9]])},
    binder_atoms_to_include=AtomFilter.BB,
)

## Simulation

The simulation configuration controls the sampler, including which solver to use (ODE/SDE) and its parameters. 

This example uses an SDE.

In [ ]:
from dynopsi.model.model_config import DEFAULT_DYNOPSI_MODEL_CONFIG
from dynopsi.simulation import LinearTimeSampler, SDESimulation
from dynopsi.utils import ConstantFunction, InverseParameterFunction

simulation = SDESimulation(
    name="sde_default",
    inference_model_config=DEFAULT_DYNOPSI_MODEL_CONFIG,
    time_sampler=LinearTimeSampler(num_steps=60),
    self_conditioning_mode="previous_step",
    diffusion_coefficient_fn=InverseParameterFunction(eps=0.01, clamp_max=10.0),
    noise_scale_fn=ConstantFunction(constant=0.3),
    ode_warmup_score_scale_fn=ConstantFunction(constant=1.0),
    ode_warmup_max_t=0.30,
    ode_cooldown_score_scale_fn=ConstantFunction(constant=1.3),
    ode_cooldown_min_t=0.87,
)

## Output Configuration
Change the output directory here as desired.

In [ ]:
from dynopsi import OutputSpecification

output_specification = OutputSpecification(
    output_dir="./example/output",
    formats=["mmcif", "npz"],
    save_trajectory=True,
)

## Putting it all together

In [ ]:
from dynopsi import DesignPipeline

design_pipeline = DesignPipeline(
    name="dynopsi_binder_design_pipeline",
    featurization_pipeline=featurization_pipeline,
    simulation_pipeline=simulation,
    output_specification=output_specification,
)

## Check Design Pipeline
Check the output directory specified in the Output Configuration step!

In [ ]:
design_pipeline.verify_configuration()

## Run Design Pipeline
Make sure that you are running sampling on a device with a CUDA-enable GPU!

Sampling a single batch with the example configuration takes ~3-4 minutes to run on one Nvidia T4 GPU with binder lengths 60-120. Each of the four binder lengths specified in the featurization configuration is sampled in its own batch, so the cell below takes ~15-16 minutes to generate samples for all lengths on a T4. To speed up the example run, either specify fewer binder lengths or reduce the number of timesteps in the time sampler input to the SDE simulation configuration. Note that reducing the number of time steps will degrade the quality of the generated structures, so this is safe if you are testing that the pipeline runs but unsafe if you are interested in generating high-quality samples.

In [ ]:
output_file_configs = design_pipeline.sample(num_samples=1)